# Model Training - ChatKasir

- Nama: Achmad Rif'an
- Bagian: AI-1 (Model Architect)

## 1. Import Library

In [18]:
import os
import time
import json
import contextlib
import numpy as np
import tensorflow as tf

from tensorflow.keras.layers import Input, Embedding, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model

## 2. Memuat Konfigurasi

Kita mengambil file model_config.json untuk mengetahui parameter model seperti vocab_size dan max_length. Ini penting agar arsitektur yang kita bangun di notebook ini sama persis dengan data yang sudah disiapkan.

In [19]:
# Memuat konfigurasi arsitektur dari file JSON
config_path = "..\\assets\\data\\model_config.json"
with open(config_path, "r") as f:
    config = json.load(f)

# Mengambil variabel penting
VOCAB_SIZE = config['vocab_size']
MAX_LENGTH = config['max_length']
NUM_TAGS = config['num_product_tags']

print(f"Konfigurasi dimuat: Vocab={VOCAB_SIZE}, Max Length={MAX_LENGTH}, Num Tags={NUM_TAGS}")

Konfigurasi dimuat: Vocab=5000, Max Length=64, Num Tags=3


## 3. Data Loading

1. Memuat Dataset: Kita memuat file dataset_chatkasir.npz yang berisi array NumPy untuk data Training, Validation, dan Testing.

2. Membuat tf.data.Dataset: Kita mengubah array tersebut menjadi objek Dataset TensorFlow. Ini adalah cara paling efisien untuk melatih model karena mendukung fitur shuffling (mengacak data) dan batching (mengambil data sedikit demi sedikit) agar tidak membebani memori RAM.

In [20]:
# Memuat dataset yang sudah dibagi (Train, Val, Test)
data_path = "..\\assets\\data\\dataset_chatkasir.npz"
data = np.load(data_path)

# Ekstrak data Training
X_train = data['X_train']
Y_prod_train = data['Y_prod_train']
Y_qty_train = data['Y_qty_train']

# normalisasi harga (dibagi 1000)
# Jika harga bukan -1, bagi dengan 1000. Jika -1, biarkan tetap -1
Y_price_train_raw = data['Y_price_train']
Y_price_train = np.where(Y_price_train_raw != -1.0, Y_price_train_raw / 1000.0, -1.0)

# Ekstrak data Validation
X_val = data['X_val']
Y_prod_val = data['Y_prod_val']
Y_qty_val = data['Y_qty_val']

# Normalisasi juga untuk data Validation
Y_price_val_raw = data['Y_price_val']
Y_price_val = np.where(Y_price_val_raw != -1.0, Y_price_val_raw / 1000.0, -1.0)

print(f"Dataset dimuat: Training={len(X_train)} baris, Validation={len(X_val)} baris")

Dataset dimuat: Training=80400 baris, Validation=10050 baris


In [21]:
# Mengonversi ke tf.data.Dataset untuk efisiensi training
BATCH_SIZE = 32 # Jumlah data yang diproses sekali epoch

def create_tf_dataset(X, y_prod, y_qty, y_price, is_training=True):
    # Gabungkan Input (X) dengan 3 Target (Y)
    ds = tf.data.Dataset.from_tensor_slices((X, (y_prod, y_qty, y_price)))
    
    if is_training:
        ds = ds.shuffle(10000) # Acak data agar model tidak menghafal urutan
    
    # Ambil data per batch dan siapkan batch berikutnya di latar belakang (prefetch)
    # AUTOTUNE = otomatis mengatur penggunaan CPU/GPU
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

# Buat objek dataset untuk Training dan Validation
train_ds = create_tf_dataset(X_train, Y_prod_train, Y_qty_train, Y_price_train)
val_ds = create_tf_dataset(X_val, Y_prod_val, Y_qty_val, Y_price_val, is_training=False)

print("Objek tf.data.Dataset berhasil dibuat")

Objek tf.data.Dataset berhasil dibuat


## 4. Re-build Model

Meskipun kita sudah merancang model di notebook sebelumnya, kita perlu mendefinisikan ulang strukturnya di notebook ini agar objek model tersebut tercipta kembali di memori sebelum dilatih.

Ada beberapa hal penting yang kita lakukan di sini:

1. Mendefinisikan TransformerEncoder: Kita menyertakan kembali kelas kustom ini, lengkap dengan dukungan masking agar model tidak "bingung" melihat token padding.

2. Mendefinisikan model_transformer: Fungsi ini membangun arsitektur Multi-Task kita yang terdiri dari satu tulang punggung (backbone) Transformer dan tiga cabang prediksi (Produk, Jumlah, Harga).

3. Instansiasi Model: Kita memanggil fungsi tersebut menggunakan variabel VOCAB_SIZE, MAX_LENGTH, dan NUM_TAGS yang sudah kita muat dari file konfigurasi di Tahap 1.

In [22]:
# Custom Layer Transformer dengan dukungan Masking
class TransformerEncoder(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super(TransformerEncoder, self).__init__(**kwargs)
        self.supports_masking = True # Pastikan layer mendukung penanda padding
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim) 
        self.ffn = tf.keras.Sequential([Dense(ff_dim, activation="relu"), Dense(embed_dim)])  
        self.layernorm1 = LayerNormalization(epsilon=1e-6) 
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)  
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training=False, mask=None):
        # Gunakan mask agar attention mengabaikan token padding [PAD]
        padding_mask = tf.cast(mask[:, tf.newaxis, :], dtype=tf.int32) if mask is not None else None
        
        attn_output = self.att(inputs, inputs, attention_mask=padding_mask)  
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)  

        ffn_output = self.ffn(out1) 
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# Fungsi arsitektur model
def model_transformer(vocab_size, max_length, num_product_tags):
    embed_dim = 64  # Dimensi representasi kata
    num_heads = 4   # Jumlah mekanisme attention
    ff_dim = 128    # Kapasitas memori internal
    
    # Input Layer
    inputs = Input(shape=(max_length,), name="input_ids")  
    
    # Shared Backbone (Transformer)
    # mask_zero=True sangat penting untuk menangani padding secara otomatis
    x = Embedding(input_dim=vocab_size, output_dim=embed_dim, mask_zero=True)(inputs)  
    x = TransformerEncoder(embed_dim, num_heads, ff_dim)(x) 
    x_pooled = GlobalAveragePooling1D()(x) # Ringkasan kalimat untuk regresi
    
    # Cabang 1: Produk (NER) - Memprediksi tag untuk setiap kata
    branch_product = Dense(64, activation='relu')(x)
    output_product = Dense(num_product_tags, activation='softmax', name="product_tags")(branch_product)
    
    # Cabang 2: Jumlah (Quantity) - Regresi nilai angka jumlah pesanan
    branch_quantity = Dense(32, activation='relu')(x_pooled)
    output_quantity = Dense(1, activation='relu', name="quantity")(branch_quantity)
    
    # Cabang 3: Harga (Price) - Regresi nilai harga satuan
    branch_price = Dense(32, activation='relu')(x_pooled)
    output_price = Dense(1, activation='relu', name="price")(branch_price)
    
    return Model(inputs=inputs, outputs=[output_product, output_quantity, output_price])

# Merakit model menggunakan parameter dari konfigurasi Tahap 1
model = model_transformer(
    vocab_size=VOCAB_SIZE,
    max_length=MAX_LENGTH,
    num_product_tags=NUM_TAGS
)

# Tampilkan ringkasan arsitektur
model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_ids           │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 64, 64)    │    320,000 │ input_ids[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 64)        │          0 │ input_ids[0][0]   │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, 64, 64)    │     83,200 │ embedding_2[0][0… │
│ (TransformerEncode… │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ transformer_enco… │
│ (GlobalAveragePool… │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 64, 64)    │      4,160 │ transformer_enco… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 32)        │      2,080 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 32)        │      2,080 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ product_tags        │ (None, 64, 3)     │        195 │ dense_12[0][0]    │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ quantity (Dense)    │ (None, 1)         │         33 │ dense_13[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ price (Dense)       │ (None, 1)         │         33 │ dense_14[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 411,781 (1.57 MB)

 Trainable params: 411,781 (1.57 MB)

 Non-trainable params: 0 (0.00 B)

## 5. Loss Function & Optimizer
1. Loss Function: Ini adalah rumus matematika untuk menghitung seberapa jauh tebakan model dari kenyataan.

   - Cabang Produk: Menggunakan Sparse Categorical Crossentropy karena tugasnya adalah klasifikasi kategori (O, B-PROD, I-PROD).

   - Cabang Quantity: Menggunakan Mean Absolute Error (MAE) karena tugasnya menebak angka kontinu.

   - Cabang Harga (MaskedPriceLoss): Ini yang paling spesial. Kita harus membuat fungsi kustom agar model mengabaikan data yang nilai harganya -1 (saat harga tidak disebutkan di chat). Jika tidak di-masking, model akan belajar menebak angka -1, padahal itu hanya penanda data kosong.

2. Optimizer: Ini adalah algoritma yang bertugas memperbaiki bobot model berdasarkan nilai loss. Kita menggunakan Adam, yang merupakan standar industri karena kecepatannya dalam belajar.

3. Dynamic Weighting Variables: Kita menyiapkan variabel pembobot awal agar nantinya model bisa menyeimbangkan fokus belajarnya antara Produk, Jumlah, dan Harga secara otomatis.

In [23]:
# Custom Loss untuk harga satuan
class MaskedPriceLoss(tf.keras.losses.Loss):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        # Menggunakan Mean Absolute Error (MAE) sebagai dasar perhitungan
        self.mae = tf.keras.losses.MeanAbsoluteError(reduction='none')

    def call(self, y_true, y_pred):
        # Buat masker: abaikan jika y_true bernilai -1
        mask = tf.cast(tf.not_equal(y_true, -1.0), tf.float32)
        
        # Hitung MAE mentah
        loss = self.mae(y_true, y_pred)
        
        # Kalikan loss dengan masker (loss jadi 0 untuk data bernilai -1)
        masked_loss = loss * mask
        
        # Kembalikan rata-rata loss hanya dari data yang valid
        return tf.reduce_sum(masked_loss) / (tf.reduce_sum(mask) + 1e-7)

# Inisialisasi Loss Function untuk tiap cabang
# Sparse karena label produk berbentuk angka ID (0, 1, 2)
loss_fn_product = tf.keras.losses.SparseCategoricalCrossentropy()
loss_fn_quantity = tf.keras.losses.MeanAbsoluteError()
loss_fn_price = MaskedPriceLoss()

# Inisialisasi Optimizer
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# Variabel untuk Dynamic Loss Weighting
# bobot awal seimbang (1.0) untuk ketiga tugas
w_product = tf.Variable(1.0, trainable=False, name="w_prod")
w_quantity = tf.Variable(1.0, trainable=False, name="w_qty")
w_price = tf.Variable(1.0, trainable=False, name="w_price")

## 6. Custom Training Loop (tf.GradientTape)

Membuat 2 fungsi utama:

1. train_step: Ini adalah rutinitas belajar model. Di sini, kita menggunakan tf.GradientTape sebagai "perekam". Saat model membuat tebakan (Forward Pass), tape merekamnya. Lalu kita hitung kesalahannya (Loss), dan kita putar balik rekaman tersebut (Backpropagation) untuk memperbaiki saraf-saraf model menggunakan Optimizer. Kita juga menerapkan Dynamic Loss Weighting dengan mengalikan loss dengan bobot w_product, w_quantity, dan w_price.

2. val_step: Ini adalah rutinitas ujian/tryout. Tidak ada tape perekam dan tidak ada perbaikan saraf. Model hanya murni menebak, lalu kita hitung seberapa meleset tebakannya.

In [24]:
# INISIALISASI METRICS
# menggunakan SparseCategoricalAccuracy untuk memantau prediksi tag produk (O, B-PROD, I-PROD)
train_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()
val_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()

# FUNGSI TRAINING
@tf.function
def train_step(x_batch, y_prod, y_qty, y_price):
    # Buka GradientTape
    with tf.GradientTape() as tape:
        
        # Forward Pass: model memprediksi
        # training=True, agar Dropout dll menyala
        pred_prod, pred_qty, pred_price = model(x_batch, training=True)
        
        # Hitung seberapa meleset prediksinya
        loss_prod = loss_fn_product(y_prod, pred_prod)
        loss_qty = loss_fn_quantity(y_qty, pred_qty)
        loss_price = loss_fn_price(y_price, pred_price)
        
        # Terapkan Dynamic Loss Weighting
        # Kalikan dengan bobot masing-masing sebelum digabungkan
        weighted_loss_prod = w_product * loss_prod
        weighted_loss_qty = w_quantity * loss_qty
        weighted_loss_price = w_price * loss_price
        
        total_loss = weighted_loss_prod + weighted_loss_qty + weighted_loss_price
        
    # Hitung gradien (arah perbaikan)
    gradients = tape.gradient(total_loss, model.trainable_weights)
    
    # Terapkan perbaikan bobot ke model menggunakan Optimizer
    optimizer.apply_gradients(zip(gradients, model.trainable_weights))

    # Update metrik akurasi training
    train_acc_metric.update_state(y_prod, pred_prod)
    
    # Kembalikan semua nilai loss untuk dipantau di layar
    return total_loss, loss_prod, loss_qty, loss_price

# FUNGSI VALIDATION
@tf.function
def val_step(x_batch, y_prod, y_qty, y_price):
    # Forward Pass saja
    # training=False, agar model memprediksi dengan kekuatan penuh (tanpa Dropout)
    pred_prod, pred_qty, pred_price = model(x_batch, training=False)
    
    # Hitung Loss mentah
    loss_prod = loss_fn_product(y_prod, pred_prod)
    loss_qty = loss_fn_quantity(y_qty, pred_qty)
    loss_price = loss_fn_price(y_price, pred_price)
    
    # Terapkan pembobotan untuk evaluasi
    weighted_loss_prod = w_product * loss_prod
    weighted_loss_qty = w_quantity * loss_qty
    weighted_loss_price = w_price * loss_price
    
    total_loss = weighted_loss_prod + weighted_loss_qty + weighted_loss_price

    # Update metrik akurasi validation
    val_acc_metric.update_state(y_prod, pred_prod)
    
    return total_loss, loss_prod, loss_qty, loss_price

print("Fungsi train_step dan val_step berhasil dibuat")

Fungsi train_step dan val_step berhasil dibuat


## 7. Model Training

Pada tahap ini, kita akan menjalankan proses pelatihan selama beberapa Epoch. Di setiap epoch, model akan melakukan serangkaian aktivitas utama:

1. Siklus Pelatihan (Training): Model membaca seluruh data di train_ds, melakukan prediksi, menghitung kesalahan, dan memperbaiki dirinya sendiri menggunakan fungsi train_step yang sudah kita buat dengan tf.GradientTape.

2. Siklus Evaluasi (Validation): Setelah satu putaran belajar selesai, model diuji menggunakan data val_ds melalui fungsi val_step untuk melihat sejauh mana ia bisa menggeneralisasi pola tanpa melakukan perbaikan bobot.

3. Penyimpanan Otomatis & Early Stopping (Custom Callback): Karena kita menggunakan Custom Training Loop, kita menerapkan logika callback manual. Sistem akan selalu memantau Validation Loss. Jika performa membaik (memecahkan rekor), model akan otomatis disimpan. Namun, jika performa stagnan atau memburuk selama beberapa putaran berturut-turut (patience), sistem akan menarik rem darurat (Early Stopping) untuk menghemat waktu dan mencegah Overfitting.

Kita juga akan memantau nilai Akurasi untuk Produk serta nilai MAE (Mean Absolute Error) dari cabang Quantity dan Price. Ini penting untuk memastikan bahwa Dynamic Loss Weighting dan MaskedPriceLoss bekerja dengan efektif dalam menyeimbangkan prioritas belajar model.

Format Ekspor Model Final:
Model dengan nilai Validation Loss terbaik akan langsung diekspor secara senyap ke dalam dua format:

1. .keras: Format Keras V3 tunggal yang ideal untuk dokumentasi (AI-1) dan eksperimen/training lanjutan di Python.

2. SavedModel (Folder): Format universal production-ready yang siap digunakan oleh tim Inference. Format inilah yang akan diserahkan kepada AI-2 (Denny) untuk di-deploy ke Server API Backend.

In [25]:
# KONFIGURASI PELATIHAN
EPOCHS = 30  # Jumlah putaran pelatihan
history = [] # Untuk menyimpan catatan progres loss

# INISIALISASI VARIABEL CUSTOM CALLBACK
best_val_loss = float('inf') # Set target awal ke angka yang sangat besar (tak terhingga)
patience = 3                 # Batas toleransi berapa epoch model boleh tidak membaik
patience_counter = 0         # Penghitung kesempatan

os.makedirs("..\\assets\\models", exist_ok=True)
save_path_keras = "..\\assets\\models\\chatkasir_model.keras"
save_path_best_sm = "..\\assets\\models\\chatkasir_saved_model"

print(f"Memulai Pelatihan selama {EPOCHS} Epoch...\n")

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # 1. TRAINING
    # Inisialisasi pengumpul loss untuk satu epoch
    epoch_train_loss = epoch_l_prod = epoch_l_qty = epoch_l_price = 0.0
    num_train_batches = 0
    
    for x_batch, (y_prod, y_qty, y_price) in train_ds:
        # Jalankan 1 langkah training
        t_loss, l_prod, l_qty, l_price = train_step(x_batch, y_prod, y_qty, y_price)
        
        # Akumulasi semua jenis loss
        epoch_train_loss += t_loss
        epoch_l_prod += l_prod
        epoch_l_qty += l_qty
        epoch_l_price += l_price
        num_train_batches += 1
        
    avg_train_loss = epoch_train_loss / num_train_batches
    avg_train_prod = epoch_l_prod / num_train_batches
    avg_train_qty = epoch_l_qty / num_train_batches
    avg_train_price = epoch_l_price / num_train_batches

    # Ambil hasil akurasi training
    train_acc = train_acc_metric.result()

    # 2. VALIDATION 
    epoch_val_loss = epoch_val_prod = epoch_val_qty = epoch_val_price = 0.0
    num_val_batches = 0
    
    for x_batch_val, (y_prod_val, y_qty_val, y_price_val) in val_ds:
        # Jalankan 1 langkah validation (tanpa update bobot)
        v_loss, v_l_prod, v_l_qty, v_l_price = val_step(x_batch_val, y_prod_val, y_qty_val, y_price_val)
        
        epoch_val_loss += v_loss
        epoch_val_prod += v_l_prod
        epoch_val_qty += v_l_qty
        epoch_val_price += v_l_price
        num_val_batches += 1
        
    avg_val_loss = epoch_val_loss / num_val_batches
    avg_val_prod = epoch_val_prod / num_val_batches
    avg_val_qty = epoch_val_qty / num_val_batches
    avg_val_price = epoch_val_price / num_val_batches

    # Ambil hasil akurasi validation
    val_acc = val_acc_metric.result()

    # 3. MONITORING LOSS & AKURASI
    duration = time.time() - start_time
    # FORMAT BARU: Menggunakan {:,.4f} untuk pemisah ribuan
    print(f"Epoch {epoch+1}/{EPOCHS} - {duration:.1f}s")
    print(f" > TOTAL Loss   : Train {avg_train_loss:.4f} | Val {avg_val_loss:.4f}")
    print(f"   > Loss Prod  : Train {avg_train_prod:.4f} | Val {avg_val_prod:.4f}")
    print(f"   > Loss Qty   : Train {avg_train_qty:.4f} | Val {avg_val_qty:.4f}")
    print(f"   > Loss Price : Train {avg_train_price:,.2f} | Val {avg_val_price:,.2f}")
    print(f" > Prod Accuracy: Train {train_acc*100:.2f}% | Val {val_acc*100:.2f}%")
    print("-" * 50)

    # LOGIKA CUSTOM CALLBACK: EARLY STOPPING & CHECKPOINT
    if avg_val_loss < best_val_loss:
        print(f"Val Loss membaik dari {best_val_loss:.4f} ke {avg_val_loss:.4f}!")
        print(f"Menyimpan model terbaik ke {save_path_best_sm}...")
        best_val_loss = avg_val_loss
        patience_counter = 0 # Reset kesabaran karena model membaik
        
        # Simpan model sebagai .keras
        model.save(save_path_keras)

        # Simpan model sebagai SavedModel
        # Belokkan semua print dari TensorFlow ke os.devnull (tempat pembuangan)
        with open(os.devnull, 'w') as f, contextlib.redirect_stdout(f):
            model.export(save_path_best_sm)

    else:
        patience_counter += 1
        print(f"Val Loss tidak membaik. Kesempatan: {patience_counter}/{patience}")
    
    print("-" * 50)
    
    # Reset metrik akurasi
    train_acc_metric.reset_state()
    val_acc_metric.reset_state()
    
    # Simpan history
    history.append({
        'train_loss': avg_train_loss.numpy(), 'val_loss': avg_val_loss.numpy(),
        'train_acc': train_acc.numpy(), 'val_acc': val_acc.numpy()
    })
    
    # Hentikan loop jika batas kesabaran habis
    if patience_counter >= patience:
        print(f"\nEARLY STOPPING TRIGGERED! Pelatihan dihentikan otomatis pada Epoch {epoch+1}.")
        print("Model sudah tidak membaik selama 3 putaran berturut-turut.")
        break

print("\nProses Pelatihan Selesai!")

Memulai Pelatihan selama 30 Epoch...

Epoch 1/30 - 72.6s
 > TOTAL Loss   : Train 5.8764 | Val 1.7524
   > Loss Prod  : Train 0.3088 | Val 0.2588
   > Loss Qty   : Train 1.3422 | Val 0.2658
   > Loss Price : Train 4.23 | Val 1.23
 > Prod Accuracy: Train 83.88% | Val 84.77%
--------------------------------------------------
Val Loss membaik dari inf ke 1.7524!
Menyimpan model terbaik ke ..\assets\models\chatkasir_saved_model...
INFO:tensorflow:Assets written to: ..\assets\models\chatkasir_saved_model\assets


INFO:tensorflow:Assets written to: ..\assets\models\chatkasir_saved_model\assets


--------------------------------------------------
Epoch 2/30 - 71.8s
 > TOTAL Loss   : Train 1.0778 | Val 1.4630
   > Loss Prod  : Train 0.2594 | Val 0.2346
   > Loss Qty   : Train 0.1864 | Val 0.1847
   > Loss Price : Train 0.63 | Val 1.04
 > Prod Accuracy: Train 84.75% | Val 85.08%
--------------------------------------------------
Val Loss membaik dari 1.7524 ke 1.4630!
Menyimpan model terbaik ke ..\assets\models\chatkasir_saved_model...
INFO:tensorflow:Assets written to: ..\assets\models\chatkasir_saved_model\assets


INFO:tensorflow:Assets written to: ..\assets\models\chatkasir_saved_model\assets


--------------------------------------------------
Epoch 3/30 - 70.2s
 > TOTAL Loss   : Train 0.9394 | Val 1.5055
   > Loss Prod  : Train 0.2400 | Val 0.2275
   > Loss Qty   : Train 0.1657 | Val 0.2119
   > Loss Price : Train 0.53 | Val 1.07
 > Prod Accuracy: Train 84.96% | Val 85.08%
--------------------------------------------------
Val Loss tidak membaik. Kesempatan: 1/3
--------------------------------------------------
Epoch 4/30 - 67.6s
 > TOTAL Loss   : Train 0.8351 | Val 1.6347
   > Loss Prod  : Train 0.2337 | Val 0.2242
   > Loss Qty   : Train 0.1498 | Val 0.3072
   > Loss Price : Train 0.45 | Val 1.10
 > Prod Accuracy: Train 85.05% | Val 85.13%
--------------------------------------------------
Val Loss tidak membaik. Kesempatan: 2/3
--------------------------------------------------
Epoch 5/30 - 68.5s
 > TOTAL Loss   : Train 0.7889 | Val 1.0724
   > Loss Prod  : Train 0.2296 | Val 0.2212
   > Loss Qty   : Train 0.1429 | Val 0.1643
   > Loss Price : Train 0.42 | Val 0.69
 > P

INFO:tensorflow:Assets written to: ..\assets\models\chatkasir_saved_model\assets


--------------------------------------------------
Epoch 6/30 - 67.7s
 > TOTAL Loss   : Train 0.7467 | Val 1.2530
   > Loss Prod  : Train 0.2263 | Val 0.2179
   > Loss Qty   : Train 0.1365 | Val 0.1641
   > Loss Price : Train 0.38 | Val 0.87
 > Prod Accuracy: Train 85.13% | Val 85.23%
--------------------------------------------------
Val Loss tidak membaik. Kesempatan: 1/3
--------------------------------------------------
Epoch 7/30 - 62.3s
 > TOTAL Loss   : Train 0.7097 | Val 1.1542
   > Loss Prod  : Train 0.2220 | Val 0.2148
   > Loss Qty   : Train 0.1296 | Val 0.1726
   > Loss Price : Train 0.36 | Val 0.77
 > Prod Accuracy: Train 85.15% | Val 85.25%
--------------------------------------------------
Val Loss tidak membaik. Kesempatan: 2/3
--------------------------------------------------
Epoch 8/30 - 62.3s
 > TOTAL Loss   : Train 0.7052 | Val 1.3184
   > Loss Prod  : Train 0.2194 | Val 0.2133
   > Loss Qty   : Train 0.1290 | Val 0.2106
   > Loss Price : Train 0.36 | Val 0.89
 > P